In [1]:
import open3d as o3d
import numpy as np
from pathlib import Path
from copy import deepcopy

def show_normals(point_cloud):
    points = np.asarray(point_cloud.points)
    normals = np.asarray(point_cloud.normals)


    vis = o3d.visualization.Visualizer()
    vis.create_window()
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.normals = o3d.utility.Vector3dVector(normals)
    vis.add_geometry(pcd)

    lines = []
    for i in range(len(points)):
        p1 = points[i]
        p2 = points[i] + normals[i] * 0.1  # 通过缩放法线长度来控制线条的长度
        line = o3d.geometry.LineSet()
        line.points = o3d.utility.Vector3dVector([p1, p2])
        line.lines = o3d.utility.Vector2iVector([[0, 1]])
        line.colors = o3d.utility.Vector3dVector([[0, 0, 1]])  # 设置线条颜色为蓝色
        lines.append(line)
        vis.add_geometry(line)
    vis.run()
    vis.destroy_window()
def icp_mesh_pcd(mesh_model, pcd):
    # criteria = o3d.registration.ICPConvergenceCriteria(max_iteration=100,
    #                                               relative_fitness=1e-6,
    #                                               relative_rmse=1e-6)
    model_point_cloud = o3d.geometry.PointCloud()
    model_point_cloud.points = mesh_model.vertices
    mesh_model.compute_vertex_normals()
    model_point_cloud.normals = mesh_model.vertex_normals

    icp_result = o3d.pipelines.registration.registration_icp(
        # 最大的crospondence距离，不能设置太小了，不然找不到对应点
        pcd, model_point_cloud, max_correspondence_distance=0.01,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(
            max_iteration=5000)
    )
    
    # cuz this is the env to icp the model
    transform = np.linalg.inv(icp_result.transformation)
    return transform

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
folder_path = Path("/media/tony/新加卷/test_data/test/test_1")
mesh_model_path = folder_path / \
    Path("object_pose_in_every_frame_with_icp/1.ply")
env_pcd_path = folder_path / Path("merged_pcd_filter/TEMP/cam0_index0.ply")
mesh_model = o3d.io.read_triangle_mesh(str(mesh_model_path))
env_pcd = o3d.io.read_point_cloud(str(env_pcd_path))
o3d.visualization.draw_geometries([mesh_model, env_pcd])
transform_matrix = icp_mesh_pcd(mesh_model, env_pcd)
temp_model = deepcopy(mesh_model)
temp_model.paint_uniform_color([0, 1, 0])
mesh_model.paint_uniform_color([1, 0, 0])
o3d.visualization.draw_geometries(
    [temp_model,mesh_model.transform(transform_matrix), env_pcd])

KeyboardInterrupt: 